# Task 2: LoRA-fine-tune Wan2.1-1.3B on Ethiopian coffee footage, then make the advertisement
Runtime > T4 GPU. Upload your raw vertical videos to Google Drive first: `MyDrive/Video_Gen/coffee_raw/`.

**1. Drive + code.** Your adapter, cache and results are saved on Drive so a session timeout does not lose them.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, glob
DRIVE = "/content/drive/MyDrive/Video_Gen"
RAW, OUT = f"{DRIVE}/coffee_raw", f"{DRIVE}/coffee_lora"
os.makedirs(RAW, exist_ok=True); os.makedirs(OUT, exist_ok=True)
print("raw videos:", sorted(os.listdir(RAW)))
from google.colab import userdata
REPO = "/content/repo"
if not os.path.exists(REPO):
    tok = userdata.get("GH_TOKEN").strip()
    assert tok.startswith(("ghp_", "github_pat_")), "GH_TOKEN must be a GitHub token"
    !git clone https://{tok}@github.com/simachewalamneh/video_gen.git {REPO}
WORK = os.path.dirname(os.path.dirname(glob.glob(f"{REPO}/**/coffee_lora/prepare_clips.py", recursive=True)[0]))
%cd {WORK}

In [ ]:
!apt-get -q install -y fonts-noto-core > /dev/null
!pip -q install -r requirements.txt
!pip -q uninstall -y torchao
!pip -q install uroman
!sed -i 's/from diffusers.utils import get_peft_model_state_dict/from peft.utils import get_peft_model_state_dict/' train_lora.py

**2. Captions.** One accurate sentence per video (what is visible: subject, action, light, camera). Edit this dictionary to match your file names.

In [ ]:
import json
PREFIX = "Ethiopian coffee commercial style: "
CAPTIONS = {
    "roasting1.mp4": "close-up of green coffee beans roasting in a metal pan over charcoal, smoke rising, warm light",
    "jebena1.mp4": "a traditional clay jebena pouring dark coffee into small white cups on a tray, steam rising",
    # "beans1.mp4": "...", "cups1.mp4": "...",
}
json.dump(CAPTIONS, open("captions.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
!python coffee_lora/prepare_clips.py --raw "{RAW}" --captions captions.json --out data/coffee --prefix "{PREFIX}"

**3. Smoke test** (about 10 minutes the first time, because it builds the cache). Check the peak GPU memory. It must stay well under 15 GB. If it runs out of memory, add `--frames 9 --height 448 --width 256` and regenerate the clips with the same numbers.

In [ ]:
!python train_lora.py --data_dir data/coffee --steps 10 --accum 1 --height 512 --width 288 --frames 17 --out "{OUT}"

**4. Train the LoRA.** The adapter is saved to Drive every 100 steps.

In [ ]:
STEPS = 400
!python train_lora.py --data_dir data/coffee --steps {STEPS} --accum 2 --rank 16 --height 512 --width 288 --frames 17 --out "{OUT}"

**5. Evaluate: base Wan vs LoRA, same prompts and seeds.** Judge whether the LoRA really improves the coffee imagery.

In [ ]:
import sys, shlex
sys.path.insert(0, "coffee_ad")
from gen_scenes import PROMPTS
args = " ".join("--prompt " + shlex.quote(PREFIX + p) for p in PROMPTS)
!python infer.py --lora_dir "{OUT}" --out compare {args} --height 512 --width 288 --frames 33

In [ ]:
from IPython.display import Video, display
for k in range(4):
    for tag in ["base", "lora"]:
        print(tag, "scene", k + 1); display(Video(f"compare/{tag}_{k}.mp4", embed=True, width=216))

**6. Make the advertisement with the LoRA.** Pick the better version per scene. Redo a weak scene with `--scenes 3 --seed 7`.

In [ ]:
!python coffee_ad/gen_scenes.py --out scenes --lora_dir "{OUT}" --prefix "{PREFIX}"
!python coffee_ad/assemble.py --scenes scenes --out ad_silent.mp4
!python coffee_ad/voiceover.py --video ad_silent.mp4 --out ethiopian_coffee_ad.mp4

In [ ]:
from IPython.display import Video, display
display(Video("ethiopian_coffee_ad.mp4", embed=True, width=270))
!cp -r compare scenes ad_silent.mp4 ethiopian_coffee_ad.mp4 voice.wav "{OUT}/"
print("saved to", OUT)